# Assignment

## Question 1

Implement RSA encryption with the given public key modulo, exponentmodulo,exponent.

You have access to the function PowMod(a, n, modulo) which computes $a^{n}$ mod *modulo* using the fast modular exponentiation algorithm from the previous module. You also have access to the function ConvertToInt(message) which converts a text message to an integer.

You need to fix the implementation of the function Encrypt(message, modulo, exponent) to return the integer ciphertext according to RSA encryption algorithm.

In [1]:
def ConvertToInt(message_str):
  res = 0
  for i in range(len(message_str)):
    res = res * 256 + ord(message_str[i])
  return res

def PowMod(a, n, mod):
    if n == 0:
        return 1 % mod
    elif n == 1:
        return a % mod
    else:
        b = PowMod(a, n // 2, mod)
        b = b * b % mod
        if n % 2 == 0:
          return b
        else:
          return b * a % mod

In [3]:
def Encrypt(message, modulo, exponent):
    message_int = ConvertToInt(message)
    return PowMod(message_int, exponent, modulo)

## Question 2

Implement RSA decryption with the given private key $p$, $q$, exponent.

You have access to the function ConvertToStr(m) which converts from integer $m$ to the plaintext message. You also have access to the function InvertModulo(a, n) which takes coprime integers $a$ and $n$ as inputs and returns integer $b$ such that $ab \equiv 1 mod\ n$. You also have access to the function PowMod(a, n, modulo) which computes $a^n$ mod *modulo* using fast modular exponentiation.

You need to fix the implementation of the function Decrypt(ciphertext, p, q, exponent) to decrypt the message which was encrypted using the public key ($n = p \cdot q$, $e$ = exponent).

In [4]:
def ConvertToStr(n):
    res = ""
    while n > 0:
        res += chr(n % 256)
        n //= 256
    return res[::-1]

def ExtendedEuclid(a, b):
    if b == 0:
        return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0:
        b = (b % n + n) % n
    return b

In [5]:
def Decrypt(ciphertext, p, q, exponent):
    modulo = p * q
    phi = (p - 1) * (q - 1)
    private_exponent = InvertModulo(exponent, phi)
    message_int = PowMod(ciphertext, private_exponent, modulo)
    return ConvertToStr(message_int)

a = 3
b = 7
c = InvertModulo(a, b)
print(c)

p = 1000000007
q = 1000000009
exponent = 23917
modulo = p * q
ciphertext = Encrypt("attack", modulo, exponent)
message = Decrypt(ciphertext, p, q, exponent)
print(message)

5
attack


## Question 3

Secret agent Alice has sent one of the following messages to the center:

1. attack
2. don't attack
3. wait

Alice has ciphered her message using public key *modulo, exponent* that is available to you, and you have intercepted her ciphertext. You want to know what was the content of her message. You have access to the function Encrypt(message, modulo, exponent) which takes in a message as a string and returns a big integer as a ciphertext. It uses RSA encryption with public key *modulo, exponent*. In the starter code, you have an example usage of the function Encrypt.

You also have function DecipherSimple(ciphertext, modulo, exponent, potential_messages) implemented in the starter code. You need to fix this implementation to solve the problem. It should take the ciphertext sent from Alice to the center, the public key modulo, exponent and the set of potential messages that Alice could have sent, and return the message that Alice encrypted and sent as a string. For example, if Alice took message "wait", encrypted it with the given modulo and exponent, and got number 139763215139763215 as the ciphertext, you will need to return the string "wait" given the ciphertext = 139763215, modulo, exponent and potential_messages = ["attack", "don't attack", "wait"].

In [6]:
def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for message in potential_messages:
        if Encrypt(message, modulo, exponent) == ciphertext:
            return message
    return "don't know"

modulo = 101
exponent = 12
ciphertext = Encrypt("attack", modulo, exponent)
print(ciphertext)
print(DecipherSimple(ciphertext, modulo, exponent, ["attack", "don't attack", "wait"]))

78
attack


## Question 4

Alice is using RSA encryption with a public key modulo, exponent such that modulo = $p \cdot q$ with one of the primes $p$ and $q$ being less than 1,000,000, and you know about it. You want to break the cipher and decrypt her message.

You can use the function Decrypt(ciphertext, p, q, e) which decrypts the ciphertext given the private key $p$, $q$ and the public exponent $e$.

You are also given the function DecipherSmallPrime(ciphertext, modulo, exponent), and you need to fix its implementation so that it can decipher the ciphertext in case when one of the prime factors of the public modulo is smaller than 1,000,000.

In [11]:
def DecipherSmallPrime(ciphertext, modulo, exponent):
    # One prime factor is guaranteed to be smaller than 1,000,000.
    for p in range(2, 1_000_000):
        if modulo % p == 0:
            q = modulo // p
            return Decrypt(ciphertext, p, q, exponent)
    return "don't know"

modulo = 101 * 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
exponent = 239
ciphertext = Encrypt("attack", modulo, exponent)
print(ciphertext)
print(DecipherSmallPrime(ciphertext, modulo, exponent))

1773488865857950936127070415107286811735036786757360876488901668828411065415319121739305112809023074111043494948978081751851444797671426477224221428915534085499185651993502038149646686545099252921036161028865989784470806731700777521041215564203613423323285066203840329997825032261425978638927438518035030191944107572667758243311046550229409822846149831396410991755012812007339251536452615590635223780672174679677196279304461931703846255925914845500897273821827255506913276266523704922524156943959725369352703672131222283682166791913172292046709244906816668026610802197814114770298014914576347693504969280551524366327904
attack


## Question 5

Alice is using RSA encryption with a public key modulo, exponent such that modulo = $p \cdot q$ with |$p$ - $q$| < 5000, and you know about it. You want to break the cipher and decrypt her message.

You have access to the function Decrypt(ciphertext, p, q, e) which decrypts the ciphertext given the private key $p$, $q$ and the public exponent $e$. You also have access to the function IntSqrt(n) which takes integer $n$ and returns the largest integer $x$ such that $x^2 \leq n$.

You are also given the function DecipherSmallDiff(ciphertext, modulo, exponent), and you need to fix its implementation so that it can decipher the ciphertext in case when the difference beteween prime factors of the public modulo is smaller than 5000.

In [8]:
def IntSqrt(n):
    # Integer square root: largest x such that x*x <= n.
    return __import__("math").isqrt(n)

def DecipherSmallDiff(ciphertext, modulo, exponent):
    # Fermat factorization: close factors are found by increasing a from sqrt(n).
    a = IntSqrt(modulo)
    if a * a < modulo:
        a += 1
    while True:
        b_squared = a * a - modulo
        b = IntSqrt(b_squared)
        if b * b == b_squared:
            p = a - b
            q = a + b
            if p > 1 and p * q == modulo:
                return Decrypt(ciphertext, p, q, exponent)
        a += 1

p = 1000000007
q = 1000000009
n = p * q
e = 239
ciphertext = Encrypt("attack", n, e)
message = DecipherSmallDiff(ciphertext, n, e)
print(ciphertext)
print(message)

48582223151298950
attack


## Question 6

You've discovered that the first prime number $p$ for the private key was generated with the same algorithm and the same random seed by two different senders Alice and Angelina due to insufficient randomness, while the second prime $q$ is different for those two private keys. You want to break both ciphers and decipher messages from both Alice and Angelina.

You are given the function Decrypt(ciphertext, p, q, e) which decrypts the ciphertext given the private key $p$, $q$ and the public exponent $e$.

You are also given the function DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent, second_ciphertext, second_modulo, second_exponent).

You need to fix its implementation so that it can decipher both the first_ciphertext and second_ciphertext in the case when first_modulo and the second_modulo share a prime factor.

In [12]:
def GCD(a, b):
    while b:
        a, b = b, a % b
    return a

def DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent,
                          second_ciphertext, second_modulo, second_exponent):
    common_prime = GCD(first_modulo, second_modulo)
    if common_prime == 1:
        return ("don't know", "don't know")

    first_other_prime = first_modulo // common_prime
    second_other_prime = second_modulo // common_prime

    first_message = Decrypt(first_ciphertext, common_prime, first_other_prime, first_exponent)
    second_message = Decrypt(second_ciphertext, common_prime, second_other_prime, second_exponent)
    return (first_message, second_message)

# Example usage with common prime p and different second primes q1 and q2
p = 101
q1 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
q2 = 1000000007
first_modulo = p * q1
second_modulo = p * q2
first_exponent = 239
second_exponent = 17
first_ciphertext = Encrypt("attack", first_modulo, first_exponent)
second_ciphertext = Encrypt("wait", second_modulo, second_exponent)
print(DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent, second_ciphertext, second_modulo, second_exponent))

('attack', 'wait')


## Question 7

Bob has sent the same message to Alice and Angelina using two different public keys (n_1, e = 2) and (n_2, e = 2) with the same exponent $e = 2$. Implement the Hastad's broadcast attack to decipher the message using the intercepted ciphertexts first_ciphertext and second_ciphertext.

You have access to the function ConvertToStr(m) which converts an integer to a plaintext message. You also have access to the function ChineseRemainderTheorem($n_1, r_1, n_2, r_2$). You also have access to the function IntSqrt(n)IntSqrt(n) which takes an integer $n$ and returns the largest integer $x$ such that $x^2 \leq n$.

Fix the implementation of the function DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo) to return the message that Bob has encrypted and sent

In [10]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    # Solve x = r1 (mod n1), x = r2 (mod n2), for coprime n1,n2.
    t = ((r2 - r1) * InvertModulo(n1, n2)) % n2
    return r1 + n1 * t

def IntSqrt(n):
    return __import__("math").isqrt(n)

def DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo):
    # For e=2, CRT reconstructs m^2 modulo n1*n2.
    combined = ChineseRemainderTheorem(
        first_modulo, first_ciphertext,
        second_modulo, second_ciphertext
    )
    message_int = IntSqrt(combined)
    if message_int * message_int != combined:
        raise ValueError("The CRT result is not an exact square; check the broadcast-attack assumptions.")
    return ConvertToStr(message_int)

p1 = 790383132652258876190399065097
q1 = 662503581792812531719955475509
p2 = 656917682542437675078478868539
q2 = 1263581691331332127259083713503
n1 = p1 * q1
n2 = p2 * q2
ciphertext1 = Encrypt("attack", n1, 2)
ciphertext2 = Encrypt("attack", n2, 2)
message = DecipherHastad(ciphertext1, n1, ciphertext2, n2)
print(message)

attack
